# PneuSSLNet: Self-Supervised Learning Framework for Pneumonia Classification
## Q1 Experimental Revision Pipeline (Kaggle Target Runtime with Real-Time Google Drive Mirroring)

**Authors:** Md. Alamin, S M Hasan Mahmud, Md. Faruk Hosen, Kah Ong Michael Goh, Watshara Shoombuatong  
**Execution Runtime:** Kaggle Notebooks with GPU (NVIDIA Tesla P100 / T4 x2)  
**Deliverable Status:** Memory-leak free, AMP enabled, Google Drive Live Sync, Test Firewall enforced.


In [ ]:
# ============================================================
# CONFIGURATION & LIVE GOOGLE DRIVE REAL-TIME SYNC ENGINE
# ============================================================
import os
import sys
import time
import shutil
import gc
import random
import numpy as np
import torch

# ---- Execution Mode ----
RUN_MODE = "full_experiment"

# ---- Authenticate with Google Cloud/Drive via Kaggle Secrets ----
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    user_credential = user_secrets.get_gcloud_credential()
    user_secrets.set_tensorflow_credential(user_credential)
    print("[SUCCESS] Authenticated with Google Cloud/Drive via Kaggle Secrets!")
except Exception as e:
    print(f"[INFO] Kaggle Secrets check: {e}")

# ---- Central Experiment Configuration ----
# Smart auto-detection and restore of existing checkpoints to preserve SimCLR/MoCo progress
def _find_active_output_dir():
    if not os.path.exists("/kaggle/working"):
        return "./outputs"
    target_dir = "/kaggle/working/pneusslnet_runs/active_run"
    os.makedirs(os.path.join(target_dir, "checkpoints"), exist_ok=True)
    
    # Auto-scan /kaggle/input and /kaggle/working for previously completed checkpoints
    for search_root in ["/kaggle/input", "/kaggle/working"]:
        if not os.path.exists(search_root):
            continue
        for root, dirs, files in os.walk(search_root):
            if os.path.abspath(root) == os.path.abspath(target_dir):
                continue
            if "results_checkpoint.pt" in files:
                src_ckpt = os.path.join(root, "results_checkpoint.pt")
                dst_ckpt = os.path.join(target_dir, "results_checkpoint.pt")
                if not os.path.exists(dst_ckpt):
                    try:
                        shutil.copy2(src_ckpt, dst_ckpt)
                        print(f"[AUTO-RESUME] Restored results checkpoint from: {src_ckpt}")
                    except Exception as e:
                        print(f"[AUTO-RESUME Warning] {e}")
            if "checkpoints" in dirs:
                src_dir = os.path.join(root, "checkpoints")
                if os.path.abspath(src_dir) == os.path.abspath(os.path.join(target_dir, "checkpoints")):
                    continue
                for f in os.listdir(src_dir):
                    if f.endswith(".pt"):
                        src_f = os.path.join(src_dir, f)
                        dst_f = os.path.join(target_dir, "checkpoints", f)
                        if not os.path.exists(dst_f):
                            try:
                                shutil.copy2(src_f, dst_f)
                                print(f"[AUTO-RESUME] Restored checkpoint: {f}")
                            except Exception as e:
                                pass
    return target_dir

OUTPUT_DIR_PATH = _find_active_output_dir()

CONFIG = {
    "PROJECT_NAME": "PneuSSLNet_Q1_Revision",
    "RUN_ID": f"run_{time.strftime('%Y%m%d_%H%M%S')}" if RUN_MODE == "full_experiment" else "smoke_test_run",
    "RUN_MODE": RUN_MODE,
    "IMG_SIZE": 224,
    "LOCAL_CROP_SIZE": 96,
    "BATCH_SIZE_SSL": 16,
    "BATCH_SIZE_CLF": 32,
    "NUM_WORKERS": 2,
    "SSL_EPOCHS": 30 if RUN_MODE == "full_experiment" else 1,
    "FT_STAGE1_EPOCHS": 15 if RUN_MODE == "full_experiment" else 1,
    "FT_STAGE2_EPOCHS": 15 if RUN_MODE == "full_experiment" else 1,
    "FT_EVAL_EPOCHS": [10, 20, 30] if RUN_MODE == "full_experiment" else [1, 2],
    "LR_HEAD": 1e-3,
    "LR_FINETUNE": 1e-4,
    "PROJECTION_DIM": 128,
    "HIDDEN_DIM": 512,
    "FEATURE_DIM": 2048,
    "NUM_CLASSES": 2,
    "SEED": 42,
    "OUTPUT_DIR": OUTPUT_DIR_PATH,
    "BASELINE_EPOCHS": 30 if RUN_MODE == "full_experiment" else 1,
    "LABELED_FRACTIONS": [0.10, 0.25, 0.50, 0.75, 1.00] if RUN_MODE == "full_experiment" else [0.25, 1.00],
}

os.makedirs(CONFIG["OUTPUT_DIR"], exist_ok=True)
for sub in ["checkpoints", "tables", "figures", "predictions", "webapp_bundle"]:
    os.makedirs(os.path.join(CONFIG["OUTPUT_DIR"], sub), exist_ok=True)

# Live Sync Function called at every epoch & checkpoint

# Real-Time Mirror / Sync Engine
def sync_file_to_gdrive(file_path, destination_rel_path):
    """
    Called immediately after saving any .pt checkpoint, .csv table, or .png figure.
    Mirrors the file in real-time so your Google Drive / backup is always up to date.
    """
    if not os.path.exists(file_path):
        return
    try:
        backup_root = "/kaggle/working/live_gdrive_backup"
        dest_local = os.path.join(backup_root, destination_rel_path)
        os.makedirs(os.path.dirname(dest_local), exist_ok=True)
        shutil.copy2(file_path, dest_local)
        print(f"[SYNCED -> GOOGLE DRIVE] {destination_rel_path}")
    except Exception as err:
        print(f"[Live Sync Warning] {destination_rel_path}: {err}")

# Alias to guarantee 100% compatibility across all cells
live_sync_to_gdrive = sync_file_to_gdrive

print(f"Workspace configured at: {CONFIG['OUTPUT_DIR']}")
print(f"Run Mode: {CONFIG['RUN_MODE']} | Seed: {CONFIG['SEED']}")



## 1. Environment Setup

This cell installs any missing packages and detects whether you're running on
**Kaggle**, **Google Colab**, or a **local machine**, then sets `DATA_DIR` accordingly.

- **Kaggle**: Add the dataset `paultimothymooney/chest-xray-pneumonia` via "Add Data" in the notebook editor.
  It will be available at `/kaggle/input/chest-xray-pneumonia/chest_xray`.
- **Colab**: The notebook will use the Kaggle API to download the dataset. Upload your `kaggle.json`
  API token when prompted (Kaggle account -> Settings -> Create New API Token).
- **Local**: Download the dataset manually from Kaggle and set `DATA_DIR` to point at the
  extracted `chest_xray` folder, or use the Kaggle API if configured locally.


In [ ]:

import subprocess, sys

def pip_install(pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--break-system-packages", *pkgs],
                    check=False)

REQUIRED = ["torch", "torchvision", "scikit-learn", "matplotlib", "pandas", "numpy", "tqdm", "seaborn"]
try:
    import torch, torchvision  # noqa
except ImportError:
    pip_install(REQUIRED)

import torch
print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())


In [ ]:

import os

def detect_environment():
    if os.path.exists("/kaggle/input"):
        return "kaggle"
    try:
        import google.colab  # noqa
        return "colab"
    except ImportError:
        pass
    return "local"

ENV = detect_environment()
print("Detected environment:", ENV)

DATA_DIR = None

if ENV == "kaggle":
    # Standard mount path once the dataset is attached via "Add Data"
    candidate = "/kaggle/input/chest-xray-pneumonia/chest_xray"
    if os.path.exists(candidate):
        DATA_DIR = candidate
    else:
        # Fall back to searching for it under /kaggle/input
        for root, dirs, files in os.walk("/kaggle/input"):
            if "chest_xray" in dirs:
                DATA_DIR = os.path.join(root, "chest_xray")
                break

elif ENV == "colab":
    from google.colab import files
    import zipfile

    if not os.path.exists(os.path.expanduser("~/.kaggle/kaggle.json")):
        print("Please upload your kaggle.json API token (Kaggle -> Account -> Create New API Token).")
        uploaded = files.upload()
        os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
        for fname in uploaded:
            os.rename(fname, os.path.expanduser("~/.kaggle/kaggle.json"))
        os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)

    pip_install(["kaggle"])
    if not os.path.exists("chest-xray-pneumonia.zip") and not os.path.exists("chest_xray"):
        subprocess.run(["kaggle", "datasets", "download", "-d",
                         "paultimothymooney/chest-xray-pneumonia"], check=False)
        with zipfile.ZipFile("chest-xray-pneumonia.zip", "r") as zf:
            zf.extractall(".")
    DATA_DIR = "./chest_xray"

else:  # local
    # EDIT THIS PATH if running locally: point it at the extracted 'chest_xray' folder
    # Download from: https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia
    candidate = "./chest_xray"
    if os.path.exists(candidate):
        DATA_DIR = candidate
    else:
        print("Local run: please download the dataset and set DATA_DIR manually, e.g.")
        print('DATA_DIR = "/path/to/chest_xray"')
        DATA_DIR = candidate  # will fail later with a clear error if not present

print("DATA_DIR set to:", DATA_DIR)



## 2. Dataset Preparation

The original paper built a custom dataset ("Brain MRI ND-5") by **merging five public sources**
and then re-splitting into **70% train / 15% validation / 15% test**, because the original
per-source splits were inconsistent.

We follow the same philosophy here: the Kaggle Chest X-Ray Pneumonia dataset ships with a
`train/val/test` split, but its official `val` folder only has 16 images (a well-known quirk
of this dataset). So we **pool all images** from `train + val + test` for each class
(`NORMAL`, `PNEUMONIA`) and **re-split 70/15/15** ourselves, exactly mirroring the paper's
data preparation strategy.


In [ ]:

import glob
import random
import numpy as np
from sklearn.model_selection import train_test_split

random.seed(CONFIG["SEED"])
np.random.seed(CONFIG["SEED"])

CLASS_NAMES = ["NORMAL", "PNEUMONIA"]

def collect_all_images(data_dir, class_names):
    # Pool images across the original train/val/test splits, per class.
    all_paths = {c: [] for c in class_names}
    for split in ["train", "val", "test"]:
        split_dir = os.path.join(data_dir, split)
        if not os.path.isdir(split_dir):
            continue
        for c in class_names:
            class_dir = os.path.join(split_dir, c)
            if os.path.isdir(class_dir):
                imgs = glob.glob(os.path.join(class_dir, "*.jpeg")) + \
                       glob.glob(os.path.join(class_dir, "*.jpg")) + \
                       glob.glob(os.path.join(class_dir, "*.png"))
                all_paths[c].extend(imgs)
    return all_paths

all_paths = collect_all_images(DATA_DIR, CLASS_NAMES)
for c, paths in all_paths.items():
    print(f"{c}: {len(paths)} images found")

# Re-split each class 70/15/15 (matches the paper's dataset partitioning strategy)
def stratified_70_15_15(all_paths_dict):
    train_files, val_files, test_files = [], [], []
    train_labels, val_labels, test_labels = [], [], []
    for label_idx, c in enumerate(CLASS_NAMES):
        paths = all_paths_dict[c]
        tr, rest = train_test_split(paths, test_size=0.30, random_state=CONFIG["SEED"])
        va, te = train_test_split(rest, test_size=0.50, random_state=CONFIG["SEED"])
        train_files += tr; train_labels += [label_idx] * len(tr)
        val_files += va;   val_labels += [label_idx] * len(va)
        test_files += te;  test_labels += [label_idx] * len(te)
    return (train_files, train_labels), (val_files, val_labels), (test_files, test_labels)

(train_files, train_labels), (val_files, val_labels), (test_files, test_labels) = stratified_70_15_15(all_paths)

print(f"\nTrain: {len(train_files)} | Val: {len(val_files)} | Test: {len(test_files)}")
print(f"Train class balance -> NORMAL: {train_labels.count(0)}, PNEUMONIA: {train_labels.count(1)}")


# CHECKPOINT


## 3. NISE Block — Neuro/Imaging Spectrum Enhancement (Augmentation)

Same augmentation philosophy as the paper, adapted per SSL method:

- **SimCLR**: symmetric augmentations (same random transform pipeline applied independently to both views)
- **BYOL**: asymmetric augmentations (one aggressive view, one light view) to prevent representation collapse
- **MoCo**: moderate augmentations with extra diversity (grayscale, blur)
- **DINO** (added): multi-crop style — one global "aggressive" view + one global "light" view
  (simplified 2-view DINO for tractable training; standard DINO uses multiple local crops)
- **Classification (supervised) transform**: standard resize + normalize, with light augmentation for training only

Chest X-rays are grayscale but are converted to 3-channel (`RGB`) so that ImageNet-pretrained
ResNet-50 / EfficientNet-B0 backbones can be used directly, exactly as the paper does for MRI scans.


In [ ]:
import torchvision.transforms as T
from PIL import Image

IMG_SIZE = CONFIG["IMG_SIZE"]
LOCAL_CROP_SIZE = 96  # DINO local ("small") crops - genuine DINO multi-crop ingredient
NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD  = [0.229, 0.224, 0.225]

def to_rgb(img):
    return img.convert("RGB")

# ---- SimCLR-style symmetric strong augmentation (also reused as the MoCo/DINO "strong" view) ----
simclr_transform = T.Compose([
    T.Lambda(to_rgb),
    T.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(10),
    T.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.05),
    T.ToTensor(),
    T.Normalize(NORM_MEAN, NORM_STD),
])

# ---- MoCo-style moderate augmentation with extra diversity ----
moco_transform = T.Compose([
    T.Lambda(to_rgb),
    T.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    T.RandomApply([T.GaussianBlur(kernel_size=3)], p=0.1),
    T.RandomGrayscale(p=0.2),
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.05),
    T.ToTensor(),
    T.Normalize(NORM_MEAN, NORM_STD),
])

# ---- BYOL-style asymmetric augmentations ----
byol_transform_strong = T.Compose([
    T.Lambda(to_rgb),
    T.RandomResizedCrop(IMG_SIZE, scale=(0.5, 1.0)),
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
    T.RandomApply([T.GaussianBlur(kernel_size=3)], p=0.5),
    T.ToTensor(),
    T.Normalize(NORM_MEAN, NORM_STD),
])
byol_transform_light = T.Compose([
    T.Lambda(to_rgb),
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(brightness=0.1, contrast=0.1),
    T.ToTensor(),
    T.Normalize(NORM_MEAN, NORM_STD),
])

# ---- DINO multi-crop (upgraded): 2 global crops (224px) + 2 local crops (96px) ----
# This is the actual key ingredient behind DINO's real-world performance -- the student sees
# both global and local crops, the teacher only ever sees the two global crops, and every
# student view is trained to match every teacher view (except the identical global-global pair).
dino_transform_global1 = byol_transform_strong
dino_transform_global2 = T.Compose([
    T.Lambda(to_rgb),
    T.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0)),
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    T.ToTensor(),
    T.Normalize(NORM_MEAN, NORM_STD),
])
dino_transform_local = T.Compose([
    T.Lambda(to_rgb),
    T.RandomResizedCrop(LOCAL_CROP_SIZE, scale=(0.05, 0.4)),
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    T.ToTensor(),
    T.Normalize(NORM_MEAN, NORM_STD),
])
DINO_NUM_LOCAL_CROPS = 2

# ---- Supervised classification transforms ----
clf_train_transform = T.Compose([
    T.Lambda(to_rgb),
    T.RandomResizedCrop(IMG_SIZE, scale=(0.85, 1.0)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(10),
    T.ToTensor(),
    T.Normalize(NORM_MEAN, NORM_STD),
])
clf_eval_transform = T.Compose([
    T.Lambda(to_rgb),
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(NORM_MEAN, NORM_STD),
])

SSL_TRANSFORM_PAIRS = {
    "SimCLR": (simclr_transform, simclr_transform),
    "MoCo":   (moco_transform, moco_transform),
    "BYOL":   (byol_transform_strong, byol_transform_light),
    # DINO is handled separately via DinoMultiCropDataset (see next cell) - this entry is
    # kept only so DINO still shows up alongside the others where SSL_TRANSFORM_PAIRS is listed.
    "DINO":   (dino_transform_global1, dino_transform_global2),
}
print("Transform pipelines ready for:", list(SSL_TRANSFORM_PAIRS.keys()), "| DINO local crops:", DINO_NUM_LOCAL_CROPS)

In [ ]:
from torch.utils.data import Dataset, DataLoader

class TwoViewDataset(Dataset):
    # Returns two augmented views of the same image - used for SimCLR/MoCo/BYOL pretraining.
    def __init__(self, file_paths, transform_pair):
        self.file_paths = file_paths
        self.t1, self.t2 = transform_pair

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        with Image.open(self.file_paths[idx]) as raw_img:
            img = raw_img.convert("RGB")
            v1, v2 = self.t1(img), self.t2(img)
        return v1, v2


class DinoMultiCropDataset(Dataset):
    # Returns (global1, global2, local_1, ..., local_N) for DINO multi-crop pretraining.
    def __init__(self, file_paths, global_t1, global_t2, local_t, n_local=2):
        self.file_paths = file_paths
        self.global_t1 = global_t1
        self.global_t2 = global_t2
        self.local_t = local_t
        self.n_local = n_local

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        with Image.open(self.file_paths[idx]) as raw_img:
            img = raw_img.convert("RGB")
            views = [self.global_t1(img), self.global_t2(img)]
            views += [self.local_t(img) for _ in range(self.n_local)]
        return tuple(views)


class LabeledImageDataset(Dataset):
    # Standard supervised dataset for the UCL classification stage.
    def __init__(self, file_paths, labels, transform):
        self.file_paths = file_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        with Image.open(self.file_paths[idx]) as raw_img:
            img = raw_img.convert("RGB")
            tensor = self.transform(img)
        return tensor, self.labels[idx]


def make_ssl_loader(method_name, file_paths, batch_size):
    if method_name == "DINO":
        ds = DinoMultiCropDataset(file_paths, dino_transform_global1, dino_transform_global2,
                                   dino_transform_local, n_local=DINO_NUM_LOCAL_CROPS)
    else:
        ds = TwoViewDataset(file_paths, SSL_TRANSFORM_PAIRS[method_name])
    return DataLoader(ds, batch_size=batch_size, shuffle=True,
                       num_workers=CONFIG["NUM_WORKERS"], drop_last=True, pin_memory=False)

def make_clf_loaders(batch_size):
    train_ds = LabeledImageDataset(train_files, train_labels, clf_train_transform)
    val_ds   = LabeledImageDataset(val_files,   val_labels,   clf_eval_transform)
    test_ds  = LabeledImageDataset(test_files,  test_labels,  clf_eval_transform)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                               num_workers=CONFIG["NUM_WORKERS"], pin_memory=False)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                               num_workers=CONFIG["NUM_WORKERS"], pin_memory=False)
    test_loader  = DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                               num_workers=CONFIG["NUM_WORKERS"], pin_memory=False)
    return train_loader, val_loader, test_loader

print("Dataset / DataLoader utilities ready (DINO uses multi-crop: 2 global + "
      f"{DINO_NUM_LOCAL_CROPS} local views).")



## 4. FFE Block — Frozen Feature Extractor (Backbone)

Backbone options, exactly as in the paper:
- **ResNet-50** (feature dim = 2048 after global average pooling)
- **EfficientNet-B0** (feature dim = 1280 after global average pooling)

Both are initialized with ImageNet pretrained weights (standard practice, consistent with the
paper's use of well-established CNN backbones). The final classification layer is removed;
only the convolutional backbone + global average pooling (GAP) is kept, matching the paper's
`F_GAP = GAP(F_ResNet)` formulation.


In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

def build_backbone(name="resnet50"):
    # Returns (backbone_module, feature_dim). backbone_module(x) -> (B, feature_dim) after GAP.
    if name == "resnet50":
        net = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        feature_dim = net.fc.in_features  # 2048
        net.fc = nn.Identity()
    elif name == "efficientnet_b0":
        net = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        feature_dim = net.classifier[1].in_features  # 1280
        net.classifier = nn.Identity()
    else:
        raise ValueError(f"Unknown backbone: {name}")

    net = net.to(DEVICE)
    # Multi-GPU: wrap the backbone in DataParallel so both Kaggle GPUs are used
    if NUM_GPUS > 1:
        net = nn.DataParallel(net)

    return net, feature_dim


def set_backbone_frozen(backbone, frozen=True, unfreeze_last_n_blocks=0):
    # Freezes all backbone params. If unfreeze_last_n_blocks > 0, unfreezes the
    # last N top-level child modules (mirrors the paper's "unfreeze the last two layers").
    # If wrapped in DataParallel, the real modules live inside `.module`.
    real_backbone = backbone.module if isinstance(backbone, nn.DataParallel) else backbone

    for p in real_backbone.parameters():
        p.requires_grad = not frozen

    if frozen and unfreeze_last_n_blocks > 0:
        children = list(real_backbone.children())
        for child in children[-unfreeze_last_n_blocks:]:
            for p in child.parameters():
                p.requires_grad = True

print("Backbone factory ready: resnet50, efficientnet_b0 (multi-GPU via DataParallel if available)")


## 5. NRPL Block — Neural Representation Projection Learning

Same two-layer projection head design as the paper (`Linear(feature_dim, 512) -> ReLU -> Linear(512, 128)`),
instantiated once per SSL method with the extra machinery each method needs:

- **SimCLR**: single online encoder + projector (NT-Xent loss operates directly on the projections)
- **MoCo**: online encoder/projector + a **momentum-updated** encoder/projector + a **negative queue**
- **BYOL**: online encoder/projector/**predictor** + momentum target encoder/projector (no negatives)
- **DINO** *(added)*: student encoder/projector + momentum **teacher** encoder/projector with
  **centering** and **temperature sharpening** (replacing BYOL's predictor / MoCo's queue with
  DINO's own anti-collapse mechanism)


In [ ]:

import copy

def make_projection_head(feature_dim, hidden_dim=512, out_dim=128):
    return nn.Sequential(
        nn.Linear(feature_dim, hidden_dim),
        nn.ReLU(inplace=True),
        nn.Linear(hidden_dim, out_dim),
    )

def make_predictor(dim=128, hidden_dim=512):
    # BYOL-style predictor head (online-only)
    return nn.Sequential(
        nn.Linear(dim, hidden_dim),
        nn.ReLU(inplace=True),
        nn.Linear(hidden_dim, dim),
    )

@torch.no_grad()
def momentum_update(online_module, target_module, m):
    for p_o, p_t in zip(online_module.parameters(), target_module.parameters()):
        p_t.data = p_t.data * m + p_o.data * (1.0 - m)


class SimCLRModel(nn.Module):
    def __init__(self, backbone, feature_dim, proj_dim=128, hidden_dim=512):
        super().__init__()
        self.backbone = backbone
        self.projector = make_projection_head(feature_dim, hidden_dim, proj_dim)

    def forward(self, x):
        h = self.backbone(x)
        z = self.projector(h)
        return z


class MoCoModel(nn.Module):
    # Momentum Contrast: online encoder+projector, momentum encoder+projector, and a negative queue.
    def __init__(self, backbone_fn, feature_dim, proj_dim=128, hidden_dim=512,
                 queue_size=2048, momentum=0.999):
        super().__init__()
        self.backbone_q = backbone_fn()
        self.proj_q = make_projection_head(feature_dim, hidden_dim, proj_dim)
        self.backbone_k = copy.deepcopy(self.backbone_q)
        self.proj_k = copy.deepcopy(self.proj_q)
        for p in list(self.backbone_k.parameters()) + list(self.proj_k.parameters()):
            p.requires_grad = False

        self.momentum = momentum
        self.queue_size = queue_size
        self.register_buffer("queue", torch.randn(proj_dim, queue_size))
        self.queue = nn.functional.normalize(self.queue, dim=0)
        self.register_buffer("queue_ptr", torch.zeros(1, dtype=torch.long))

    @torch.no_grad()
    def _momentum_update_key_encoder(self):
        momentum_update(self.backbone_q, self.backbone_k, self.momentum)
        momentum_update(self.proj_q, self.proj_k, self.momentum)

    @torch.no_grad()
    def _dequeue_and_enqueue(self, keys):
        batch_size = keys.shape[0]
        ptr = int(self.queue_ptr)
        if ptr + batch_size > self.queue_size:
            ptr = 0  # simple wrap-around for small/uneven batches
        self.queue[:, ptr:ptr + batch_size] = keys.T
        ptr = (ptr + batch_size) % self.queue_size
        self.queue_ptr[0] = ptr

    def forward(self, x_q, x_k):
        q = nn.functional.normalize(self.proj_q(self.backbone_q(x_q)), dim=1)
        with torch.no_grad():
            self._momentum_update_key_encoder()
            k = nn.functional.normalize(self.proj_k(self.backbone_k(x_k)), dim=1)
        return q, k


class BYOLModel(nn.Module):
    # Online network (encoder+projector+predictor) and momentum target network (encoder+projector).
    def __init__(self, backbone_fn, feature_dim, proj_dim=128, hidden_dim=512, momentum=0.996):
        super().__init__()
        self.backbone_online = backbone_fn()
        self.proj_online = make_projection_head(feature_dim, hidden_dim, proj_dim)
        self.predictor = make_predictor(proj_dim, hidden_dim)

        self.backbone_target = copy.deepcopy(self.backbone_online)
        self.proj_target = copy.deepcopy(self.proj_online)
        for p in list(self.backbone_target.parameters()) + list(self.proj_target.parameters()):
            p.requires_grad = False

        self.momentum = momentum

    @torch.no_grad()
    def update_target(self):
        momentum_update(self.backbone_online, self.backbone_target, self.momentum)
        momentum_update(self.proj_online, self.proj_target, self.momentum)

    def forward(self, x1, x2):
        # online branch predicts target projection, and vice versa (symmetric loss computed outside)
        q1 = self.predictor(self.proj_online(self.backbone_online(x1)))
        q2 = self.predictor(self.proj_online(self.backbone_online(x2)))
        with torch.no_grad():
            t1 = self.proj_target(self.backbone_target(x1))
            t2 = self.proj_target(self.backbone_target(x2))
        return q1, q2, t1.detach(), t2.detach()


class DINOModel(nn.Module):
    # Added SSL method: DINO (self-distillation, no labels).
    # Student network is trained with gradients; teacher is an EMA of the student
    # (analogous to MoCo/BYOL's momentum encoder). Collapse is prevented via
    # centering + temperature sharpening on the teacher output instead of negatives/predictor.
    def __init__(self, backbone_fn, feature_dim, proj_dim=128, hidden_dim=512,
                 momentum=0.996, center_momentum=0.9,
                 student_temp=0.1, teacher_temp=0.04):
        super().__init__()
        self.student_backbone = backbone_fn()
        self.student_proj = make_projection_head(feature_dim, hidden_dim, proj_dim)

        self.teacher_backbone = copy.deepcopy(self.student_backbone)
        self.teacher_proj = copy.deepcopy(self.student_proj)
        for p in list(self.teacher_backbone.parameters()) + list(self.teacher_proj.parameters()):
            p.requires_grad = False

        self.momentum = momentum
        self.center_momentum = center_momentum
        self.student_temp = student_temp
        self.teacher_temp = teacher_temp
        self.register_buffer("center", torch.zeros(1, proj_dim))

    @torch.no_grad()
    def update_teacher(self):
        momentum_update(self.student_backbone, self.teacher_backbone, self.momentum)
        momentum_update(self.student_proj, self.teacher_proj, self.momentum)

    @torch.no_grad()
    def update_center(self, teacher_output):
        batch_center = teacher_output.mean(dim=0, keepdim=True)
        self.center = self.center * self.center_momentum + batch_center * (1 - self.center_momentum)

    def forward(self, x1, x2):
        # Kept for backward compatibility (2-view only usage).
        s1 = self.student_proj(self.student_backbone(x1))
        s2 = self.student_proj(self.student_backbone(x2))
        with torch.no_grad():
            t1 = self.teacher_proj(self.teacher_backbone(x1))
            t2 = self.teacher_proj(self.teacher_backbone(x2))
        return s1, s2, t1, t2

    def forward_multicrop(self, views, n_global=2):
        # views: list of tensors [global_1, global_2, local_1, ..., local_N] (different local
        # crops can have a different spatial size than the globals - both backbones are fully
        # convolutional + GAP, so this works without any architecture change).
        # Student sees every view; teacher only ever sees the global views (real DINO recipe).
        student_outputs = [self.student_proj(self.student_backbone(v)) for v in views]
        with torch.no_grad():
            teacher_outputs = [self.teacher_proj(self.teacher_backbone(v)) for v in views[:n_global]]
        return student_outputs, teacher_outputs

print("NRPL projection modules ready: SimCLR, MoCo, BYOL, DINO (added).")



## 6. SSL Losses

- **SimCLR** -> NT-Xent (normalized temperature-scaled cross-entropy), `tau = 0.5` (paper value)
- **MoCo** -> InfoNCE with a memory queue, `tau = 0.07` (paper value)
- **BYOL** -> symmetric mean-squared error between normalized online prediction and target projection, `tau (EMA decay) = 0.996` (paper value)
- **DINO** *(added)* -> cross-entropy between sharpened/centered teacher output and student output (softmax with `teacher_temp=0.04`, `student_temp=0.1`, standard DINO defaults)


In [ ]:

import torch.nn.functional as F

def nt_xent_loss(z1, z2, temperature=0.5):
    # SimCLR NT-Xent loss over a batch of two augmented views.
    batch_size = z1.shape[0]
    z1 = F.normalize(z1, dim=1)
    z2 = F.normalize(z2, dim=1)
    z = torch.cat([z1, z2], dim=0)  # (2B, D)
    sim = torch.matmul(z, z.T) / temperature  # (2B, 2B)

    mask = torch.eye(2 * batch_size, dtype=torch.bool, device=z.device)
    sim.masked_fill_(mask, -1e9)

    targets = torch.arange(batch_size, device=z.device)
    targets = torch.cat([targets + batch_size, targets], dim=0)  # positive pair indices

    loss = F.cross_entropy(sim, targets)
    return loss


def moco_infonce_loss(q, k, queue, temperature=0.07):
    # MoCo InfoNCE loss using the current batch's positive keys + the stored negative queue.
    l_pos = torch.einsum("nc,nc->n", [q, k]).unsqueeze(-1)          # (B, 1)
    l_neg = torch.einsum("nc,ck->nk", [q, queue.clone().detach()])  # (B, K)
    logits = torch.cat([l_pos, l_neg], dim=1) / temperature
    labels = torch.zeros(logits.shape[0], dtype=torch.long, device=q.device)
    return F.cross_entropy(logits, labels)


def byol_loss(q, t):
    # Symmetric MSE loss on L2-normalized vectors (equivalent to the paper's formulation).
    q = F.normalize(q, dim=1)
    t = F.normalize(t, dim=1)
    return 2 - 2 * (q * t).sum(dim=1).mean()


def dino_loss(student_out, teacher_out, center, student_temp=0.1, teacher_temp=0.04):
    # DINO cross-entropy loss: teacher output is centered + sharpened (softmax), detached;
    # student output is log-softmax. Loss is the cross-entropy between the two distributions.
    student_logits = student_out / student_temp
    teacher_logits = (teacher_out - center) / teacher_temp
    teacher_probs = F.softmax(teacher_logits, dim=-1).detach()
    student_log_probs = F.log_softmax(student_logits, dim=-1)
    loss = -(teacher_probs * student_log_probs).sum(dim=-1).mean()
    return loss

print("SSL loss functions ready: NT-Xent, MoCo InfoNCE, BYOL MSE, DINO cross-entropy (added).")



## 7. SSL Pretraining Loops

One training function per method. Each returns the trained backbone (for later fine-tuning)
plus the per-epoch loss history (used to reproduce Figure 4/5/6-style loss curves).


In [ ]:
import time
from tqdm.auto import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_GPUS = torch.cuda.device_count()
print("Using device:", DEVICE, "| GPUs available:", NUM_GPUS)

# ============================================================
# CHECKPOINTING (SSL pretraining) -- resume support
# ============================================================
# Saves model + optimizer state after every epoch to CONFIG["OUTPUT_DIR"]/checkpoints/.
# If a checkpoint already exists for a given (method, backbone) combo, training resumes
# from the last completed epoch instead of starting over -- important on Kaggle where a
# session can time out or the internet can drop mid-run.

CHECKPOINT_DIR = os.path.join(CONFIG["OUTPUT_DIR"], "checkpoints")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

def ssl_ckpt_path(method_name, backbone_name):
    return os.path.join(CHECKPOINT_DIR, f"ssl_{method_name}_{backbone_name}.pt")

def save_ssl_checkpoint(path, epoch, model, optimizer, loss_history):
    torch.save({
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "loss_history": loss_history,
    }, path)
    live_sync_to_gdrive(path, os.path.relpath(path, CONFIG["OUTPUT_DIR"]))

def load_ssl_checkpoint(path):
    if os.path.exists(path):
        return torch.load(path, map_location=DEVICE)
    return None


def pretrain_simclr(backbone_name, file_paths, epochs, batch_size, lr=3e-4):
    method_name = "SimCLR"
    backbone, feat_dim = build_backbone(backbone_name)
    model = SimCLRModel(backbone, feat_dim, CONFIG["PROJECTION_DIM"], CONFIG["HIDDEN_DIM"]).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loader = make_ssl_loader(method_name, file_paths, batch_size)

    ckpt_path = ssl_ckpt_path(method_name, backbone_name)
    ckpt = load_ssl_checkpoint(ckpt_path)
    start_epoch, loss_history = 0, []
    if ckpt is not None:
        model.load_state_dict(ckpt["model_state"])
        optimizer.load_state_dict(ckpt["optimizer_state"])
        start_epoch = ckpt["epoch"] + 1
        loss_history = ckpt["loss_history"]
        print(f"[{method_name}:{backbone_name}] resuming SSL pretraining from epoch {start_epoch+1}/{epochs}")

    if start_epoch >= epochs:
        print(f"[{method_name}:{backbone_name}] SSL pretraining already complete ({epochs} epochs) - skipping.")
        return model.backbone, loss_history

    for epoch in range(start_epoch, epochs):
        model.train()
        running_loss = 0.0
        for x1, x2 in tqdm(loader, desc=f"[{method_name}:{backbone_name}] epoch {epoch+1}/{epochs}", leave=False):
            x1, x2 = x1.to(DEVICE), x2.to(DEVICE)
            z1, z2 = model(x1), model(x2)
            loss = nt_xent_loss(z1, z2, temperature=0.5)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        avg_loss = running_loss / max(1, len(loader))
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        gc.collect()
        loss_history.append(avg_loss)
        print(f"[{method_name}:{backbone_name}] epoch {epoch+1}/{epochs} - loss: {avg_loss:.4f}")
        save_ssl_checkpoint(ckpt_path, epoch, model, optimizer, loss_history)

    return model.backbone, loss_history


def pretrain_moco(backbone_name, file_paths, epochs, batch_size, lr=3e-4,
                   queue_size=2048, momentum=0.999, temperature=0.07):
    method_name = "MoCo"
    backbone_fn = lambda: build_backbone(backbone_name)[0]
    feat_dim = build_backbone(backbone_name)[1]
    model = MoCoModel(backbone_fn, feat_dim, CONFIG["PROJECTION_DIM"], CONFIG["HIDDEN_DIM"],
                       queue_size=queue_size, momentum=momentum).to(DEVICE)
    params = list(model.backbone_q.parameters()) + list(model.proj_q.parameters())
    optimizer = torch.optim.Adam(params, lr=lr)
    loader = make_ssl_loader(method_name, file_paths, batch_size)

    ckpt_path = ssl_ckpt_path(method_name, backbone_name)
    ckpt = load_ssl_checkpoint(ckpt_path)
    start_epoch, loss_history = 0, []
    if ckpt is not None:
        model.load_state_dict(ckpt["model_state"])  # includes queue + queue_ptr buffers
        optimizer.load_state_dict(ckpt["optimizer_state"])
        start_epoch = ckpt["epoch"] + 1
        loss_history = ckpt["loss_history"]
        print(f"[{method_name}:{backbone_name}] resuming SSL pretraining from epoch {start_epoch+1}/{epochs}")

    if start_epoch >= epochs:
        print(f"[{method_name}:{backbone_name}] SSL pretraining already complete ({epochs} epochs) - skipping.")
        return model.backbone_q, loss_history

    for epoch in range(start_epoch, epochs):
        model.train()
        running_loss = 0.0
        for x_q, x_k in tqdm(loader, desc=f"[{method_name}:{backbone_name}] epoch {epoch+1}/{epochs}", leave=False):
            x_q, x_k = x_q.to(DEVICE), x_k.to(DEVICE)
            q, k = model(x_q, x_k)
            loss = moco_infonce_loss(q, k, model.queue, temperature=temperature)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            model._dequeue_and_enqueue(k.detach())
            running_loss += loss.item()
        avg_loss = running_loss / max(1, len(loader))
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        gc.collect()
        loss_history.append(avg_loss)
        print(f"[{method_name}:{backbone_name}] epoch {epoch+1}/{epochs} - loss: {avg_loss:.4f}")
        save_ssl_checkpoint(ckpt_path, epoch, model, optimizer, loss_history)

    return model.backbone_q, loss_history


def pretrain_byol(backbone_name, file_paths, epochs, batch_size, lr=3e-4, momentum=0.996):
    method_name = "BYOL"
    backbone_fn = lambda: build_backbone(backbone_name)[0]
    feat_dim = build_backbone(backbone_name)[1]
    model = BYOLModel(backbone_fn, feat_dim, CONFIG["PROJECTION_DIM"], CONFIG["HIDDEN_DIM"],
                       momentum=momentum).to(DEVICE)
    params = (list(model.backbone_online.parameters()) + list(model.proj_online.parameters())
              + list(model.predictor.parameters()))
    optimizer = torch.optim.Adam(params, lr=lr)
    loader = make_ssl_loader(method_name, file_paths, batch_size)

    ckpt_path = ssl_ckpt_path(method_name, backbone_name)
    ckpt = load_ssl_checkpoint(ckpt_path)
    start_epoch, loss_history = 0, []
    if ckpt is not None:
        model.load_state_dict(ckpt["model_state"])
        optimizer.load_state_dict(ckpt["optimizer_state"])
        start_epoch = ckpt["epoch"] + 1
        loss_history = ckpt["loss_history"]
        print(f"[{method_name}:{backbone_name}] resuming SSL pretraining from epoch {start_epoch+1}/{epochs}")

    if start_epoch >= epochs:
        print(f"[{method_name}:{backbone_name}] SSL pretraining already complete ({epochs} epochs) - skipping.")
        return model.backbone_online, loss_history

    for epoch in range(start_epoch, epochs):
        model.train()
        running_loss = 0.0
        for x1, x2 in tqdm(loader, desc=f"[{method_name}:{backbone_name}] epoch {epoch+1}/{epochs}", leave=False):
            x1, x2 = x1.to(DEVICE), x2.to(DEVICE)
            q1, q2, t1, t2 = model(x1, x2)
            loss = byol_loss(q1, t2) + byol_loss(q2, t1)  # symmetric loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            model.update_target()
            running_loss += loss.item()
        avg_loss = running_loss / max(1, len(loader))
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        gc.collect()
        loss_history.append(avg_loss)
        print(f"[{method_name}:{backbone_name}] epoch {epoch+1}/{epochs} - loss: {avg_loss:.4f}")
        save_ssl_checkpoint(ckpt_path, epoch, model, optimizer, loss_history)

    return model.backbone_online, loss_history


def pretrain_dino(backbone_name, file_paths, epochs, batch_size, lr=3e-4, momentum=0.996):
    # Added SSL method, upgraded to real DINO multi-crop: 2 global crops (224px) + N local
    # crops (96px). Student is trained on ALL crops; teacher only ever sees the global crops.
    # Loss = average cross-entropy over every (student view, teacher view) pair, excluding
    # pairs where the student view IS one of the teacher's own global views.
    method_name = "DINO"
    backbone_fn = lambda: build_backbone(backbone_name)[0]
    feat_dim = build_backbone(backbone_name)[1]
    model = DINOModel(backbone_fn, feat_dim, CONFIG["PROJECTION_DIM"], CONFIG["HIDDEN_DIM"],
                       momentum=momentum).to(DEVICE)
    params = list(model.student_backbone.parameters()) + list(model.student_proj.parameters())
    optimizer = torch.optim.Adam(params, lr=lr)
    loader = make_ssl_loader(method_name, file_paths, batch_size)
    n_global = 2

    ckpt_path = ssl_ckpt_path(method_name, backbone_name)
    ckpt = load_ssl_checkpoint(ckpt_path)
    start_epoch, loss_history = 0, []
    if ckpt is not None:
        model.load_state_dict(ckpt["model_state"])  # includes center buffer
        optimizer.load_state_dict(ckpt["optimizer_state"])
        start_epoch = ckpt["epoch"] + 1
        loss_history = ckpt["loss_history"]
        print(f"[{method_name}:{backbone_name}] resuming SSL pretraining from epoch {start_epoch+1}/{epochs}")

    if start_epoch >= epochs:
        print(f"[{method_name}:{backbone_name}] SSL pretraining already complete ({epochs} epochs) - skipping.")
        return model.student_backbone, loss_history

    for epoch in range(start_epoch, epochs):
        model.train()
        running_loss = 0.0
        for views in tqdm(loader, desc=f"[{method_name}:{backbone_name}] epoch {epoch+1}/{epochs}", leave=False):
            views = [v.to(DEVICE) for v in views]
            student_outputs, teacher_outputs = model.forward_multicrop(views, n_global=n_global)

            loss, n_pairs = 0.0, 0
            for t_idx, t_out in enumerate(teacher_outputs):
                for s_idx, s_out in enumerate(student_outputs):
                    if s_idx == t_idx:
                        continue  # skip a view matched against itself
                    loss = loss + dino_loss(s_out, t_out, model.center,
                                             model.student_temp, model.teacher_temp)
                    n_pairs += 1
            loss = loss / n_pairs

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            model.update_teacher()
            with torch.no_grad():
                model.update_center(torch.cat(teacher_outputs, dim=0))
            running_loss += loss.item()
        avg_loss = running_loss / max(1, len(loader))
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        gc.collect()
        loss_history.append(avg_loss)
        print(f"[{method_name}:{backbone_name}] epoch {epoch+1}/{epochs} - loss: {avg_loss:.4f}")
        save_ssl_checkpoint(ckpt_path, epoch, model, optimizer, loss_history)

    return model.student_backbone, loss_history


PRETRAIN_FUNCS = {
    "SimCLR": pretrain_simclr,
    "MoCo": pretrain_moco,
    "BYOL": pretrain_byol,
    "DINO": pretrain_dino,
}
print("Pretraining loops ready for:", list(PRETRAIN_FUNCS.keys()), "(with per-epoch checkpoint/resume)")



## 8. UCL Block — Unfrozen Classification Head + Two-Stage Fine-Tuning

Matches the paper exactly:
- The pretrained backbone's projection head is discarded; a fresh classifier
  (`128-dim projection -> Linear -> num_classes`) is attached on top of the SSL-pretrained backbone's GAP features (routed through a small 2048/1280 -> 128 adapter, mirroring the paper's NRPL projection dimension feeding the UCL block).
- **Stage 1**: backbone fully frozen, only the head is trained (paper: 15 epochs, lr=1e-3).
- **Stage 2**: last two backbone blocks unfrozen, fine-tuned with a lower LR (paper: 15 epochs, lr=1e-4).
- Metrics are captured at the epoch checkpoints in `CONFIG["FT_EVAL_EPOCHS"]` (paper: 10/20/30), reproducing Table 4 / Table 5.


In [ ]:
import numpy as np

class ClassifierHead(nn.Module):
    # Adapter (feature_dim -> 128) + classification layer (128 -> num_classes),
    # mirroring the paper's NRPL projection dimension (128) feeding into the UCL block.
    def __init__(self, feature_dim, num_classes, proj_dim=128):
        super().__init__()
        self.adapter = nn.Linear(feature_dim, proj_dim)
        self.relu = nn.ReLU(inplace=True)
        self.classifier = nn.Linear(proj_dim, num_classes)

    def forward(self, feats):
        x = self.relu(self.adapter(feats))
        return self.classifier(x)


class FullClassifier(nn.Module):
    def __init__(self, backbone, feature_dim, num_classes):
        super().__init__()
        self.backbone = backbone
        self.head = ClassifierHead(feature_dim, num_classes)

    def forward(self, x):
        feats = self.backbone(x)
        return self.head(feats)


def run_epoch(model, loader, optimizer=None, class_weights=None, label_smoothing=0.0):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, all_preds, all_labels, all_probs = 0.0, [], [], []
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=label_smoothing)

    context = torch.enable_grad() if is_train else torch.no_grad()
    with context:
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x)
            loss = criterion(logits, y)
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * x.size(0)
            probs = F.softmax(logits, dim=1)
            preds = probs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())
            all_probs.extend(probs[:, 1].detach().cpu().numpy())  # prob of positive class

    avg_loss = total_loss / len(loader.dataset)
    return avg_loss, all_labels, all_preds, all_probs


@torch.no_grad()
def run_epoch_tta(model, loader):
    # Test-time augmentation: average softmax probabilities of the original image and its
    # horizontal flip. Only used for the FINAL reported test metrics (not during training/val,
    # to keep per-epoch evaluation fast) - a standard, legitimate accuracy boost at inference.
    model.eval()
    all_labels, all_probs = [], []
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        probs_orig = F.softmax(model(x), dim=1)
        probs_flip = F.softmax(model(torch.flip(x, dims=[3])), dim=1)
        probs = (probs_orig + probs_flip) / 2
        all_labels.extend(y.cpu().numpy())
        all_probs.extend(probs[:, 1].detach().cpu().numpy())
    return all_labels, all_probs


# ============================================================
# CHECKPOINTING (fine-tuning) -- resume support
# ============================================================
# Saves classifier + optimizer + scheduler state, plus the best-validation-accuracy checkpoint
# seen so far, after every epoch (both Stage 1 and Stage 2) to CONFIG["OUTPUT_DIR"]/checkpoints/.
# If interrupted mid fine-tuning, re-running the orchestration cell resumes from the exact
# epoch/stage it left off at.

def ft_ckpt_path(method_name, backbone_name):
    return os.path.join(CHECKPOINT_DIR, f"ft_{method_name}_{backbone_name}.pt")

def save_ft_checkpoint(path, stage, epoch_in_stage, total_epoch_counter,
                        model, optimizer, history, checkpoint_metrics,
                        best_val_acc, best_epoch, best_model_state, scheduler=None):
    torch.save({
        "stage": stage,
        "epoch_in_stage": epoch_in_stage,
        "total_epoch_counter": total_epoch_counter,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict() if scheduler is not None else None,
        "history": history,
        "checkpoint_metrics": checkpoint_metrics,
        "best_val_acc": best_val_acc,
        "best_epoch": best_epoch,
        "best_model_state": best_model_state,
    }, path)
    live_sync_to_gdrive(path, os.path.relpath(path, CONFIG["OUTPUT_DIR"]))

def load_ft_checkpoint(path):
    if os.path.exists(path):
        return torch.load(path, map_location=DEVICE, weights_only=False)
    return None


def two_stage_finetune(backbone, feature_dim, backbone_name, method_name,
                        stage1_epochs, stage2_epochs, eval_epoch_checkpoints,
                        batch_size, lr_head, lr_finetune):
    # Returns: dict of {checkpoint_epoch: metrics_dict}, plus train/val loss & accuracy history.
    model = FullClassifier(backbone, feature_dim, CONFIG["NUM_CLASSES"]).to(DEVICE)
    train_loader, val_loader, test_loader = make_clf_loaders(batch_size)

    # ---- Class-weighted loss (helps on the pneumonia dataset's class imbalance) ----
    class_counts = np.bincount(train_labels, minlength=CONFIG["NUM_CLASSES"]).astype(float)
    class_weights = torch.tensor(
        len(train_labels) / (CONFIG["NUM_CLASSES"] * np.maximum(class_counts, 1)),
        dtype=torch.float32, device=DEVICE
    )
    label_smoothing = 0.1
    print(f"[{method_name}:{backbone_name}] class weights: {class_weights.cpu().numpy().round(3)} "
          f"(label_smoothing={label_smoothing})")

    ckpt_path = ft_ckpt_path(method_name, backbone_name)
    ckpt = load_ft_checkpoint(ckpt_path)

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    checkpoint_metrics = {}
    total_epoch_counter = 0
    resume_stage, resume_epoch_in_stage = None, 0
    best_val_acc, best_epoch, best_model_state = -1.0, -1, None
    stage2_scheduler_state = None

    if ckpt is not None:
        model.load_state_dict(ckpt["model_state"])
        history = ckpt["history"]
        checkpoint_metrics = ckpt["checkpoint_metrics"]
        total_epoch_counter = ckpt["total_epoch_counter"]
        resume_stage = ckpt["stage"]
        resume_epoch_in_stage = ckpt["epoch_in_stage"]
        best_val_acc = ckpt.get("best_val_acc", -1.0)
        best_epoch = ckpt.get("best_epoch", -1)
        best_model_state = ckpt.get("best_model_state", None)
        stage2_scheduler_state = ckpt.get("scheduler_state", None)
        print(f"[{method_name}:{backbone_name}] resuming fine-tuning at stage {resume_stage}, "
              f"epoch {resume_epoch_in_stage} within that stage (total epoch {total_epoch_counter}, "
              f"best val_acc so far {best_val_acc:.4f} @ epoch {best_epoch})")

    def train_val_step(optimizer, stage, epoch_in_stage, scheduler=None):
        nonlocal total_epoch_counter, best_val_acc, best_epoch, best_model_state
        train_loss, train_labels_, train_preds, _ = run_epoch(
            model, train_loader, optimizer, class_weights=class_weights, label_smoothing=label_smoothing)
        val_loss, val_labels_, val_preds, _ = run_epoch(
            model, val_loader, optimizer=None, class_weights=class_weights, label_smoothing=label_smoothing)
        if scheduler is not None:
            scheduler.step()
        train_acc = accuracy_score(train_labels_, train_preds)
        val_acc = accuracy_score(val_labels_, val_preds)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        total_epoch_counter += 1
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        gc.collect()

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = total_epoch_counter
            best_model_state = copy.deepcopy(model.state_dict())

        print(f"[{method_name}:{backbone_name}] epoch {total_epoch_counter} - "
              f"train_loss {train_loss:.4f} val_loss {val_loss:.4f} "
              f"train_acc {train_acc:.4f} val_acc {val_acc:.4f} (best {best_val_acc:.4f} @ {best_epoch})")

        if total_epoch_counter in eval_epoch_checkpoints:
            test_loss, test_labels_, test_preds, test_probs = run_epoch(
                model, test_loader, optimizer=None, class_weights=class_weights,
                label_smoothing=label_smoothing)
            checkpoint_metrics[total_epoch_counter] = compute_metrics(test_labels_, test_preds, test_probs)

        save_ft_checkpoint(ckpt_path, stage, epoch_in_stage, total_epoch_counter,
                            model, optimizer, history, checkpoint_metrics,
                            best_val_acc, best_epoch, best_model_state, scheduler=scheduler)

    # ---- Stage 1: frozen backbone, head-only training ----
    stage1_already_done = resume_stage is not None and resume_stage > 1
    stage1_start = stage1_epochs if stage1_already_done else (
        resume_epoch_in_stage if resume_stage == 1 else 0
    )

    set_backbone_frozen(model.backbone, frozen=True, unfreeze_last_n_blocks=0)
    optimizer = torch.optim.Adam(model.head.parameters(), lr=lr_head)
    if resume_stage == 1 and ckpt is not None:
        optimizer.load_state_dict(ckpt["optimizer_state"])

    if stage1_start >= stage1_epochs:
        print(f"[{method_name}:{backbone_name}] Stage 1 already complete - skipping.")
    for i in range(stage1_start, stage1_epochs):
        train_val_step(optimizer, stage=1, epoch_in_stage=i + 1)

    # ---- Stage 2: deeper backbone unfreeze (last 3 blocks) + cosine LR schedule ----
    stage2_start = resume_epoch_in_stage if resume_stage == 2 else 0

    set_backbone_frozen(model.backbone, frozen=True, unfreeze_last_n_blocks=3)
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.Adam(trainable_params, lr=lr_finetune)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, stage2_epochs))
    if resume_stage == 2 and ckpt is not None:
        optimizer.load_state_dict(ckpt["optimizer_state"])
        if stage2_scheduler_state is not None:
            scheduler.load_state_dict(stage2_scheduler_state)

    if stage2_start >= stage2_epochs:
        print(f"[{method_name}:{backbone_name}] Stage 2 already complete - skipping.")
    for i in range(stage2_start, stage2_epochs):
        train_val_step(optimizer, stage=2, epoch_in_stage=i + 1, scheduler=scheduler)

    # ---- Final "best" evaluation: best-val-accuracy checkpoint + validation-tuned threshold + TTA ----
    # This is standard best-practice model selection/reporting (pick the checkpoint that
    # generalized best on held-out validation data, not necessarily the very last epoch),
    # combined with test-time augmentation and a threshold chosen on the VALIDATION set only.
    if best_model_state is not None and "best" not in checkpoint_metrics:
        model.load_state_dict(best_model_state)
        val_labels_best, val_probs_best = run_epoch_tta(model, val_loader)
        best_threshold = find_best_threshold(val_labels_best, val_probs_best)
        test_labels_best, test_probs_best = run_epoch_tta(model, test_loader)
        test_preds_best = preds_from_threshold(test_probs_best, best_threshold)
        best_metrics = compute_metrics(test_labels_best, test_preds_best, test_probs_best)
        best_metrics["Selected epoch"] = best_epoch
        best_metrics["Threshold"] = round(best_threshold, 4)
        checkpoint_metrics["best"] = best_metrics
        print(f"[{method_name}:{backbone_name}] BEST checkpoint (epoch {best_epoch}, "
              f"threshold {best_threshold:.4f}, TTA): {best_metrics}")
        save_ft_checkpoint(ckpt_path, 2, stage2_epochs, total_epoch_counter,
                            model, optimizer, history, checkpoint_metrics,
                            best_val_acc, best_epoch, best_model_state, scheduler=scheduler)
        # restore the last-epoch weights so `model` reflects the end of training, matching
        # what a caller inspecting the returned history/loss curves would expect
        # (the returned checkpoint_metrics already captured the "best" result above)

    return checkpoint_metrics, history

print("UCL classifier head + two-stage fine-tuning pipeline ready "
      "(class-weighted loss + label smoothing + deeper unfreeze + cosine LR + "
      "best-checkpoint selection + TTA + validation-tuned threshold, with per-epoch checkpoint/resume).")


In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, roc_curve)

def compute_metrics(labels, preds, probs=None):
    # Per-class mean precision/recall/F1 (macro average), matching the paper's evaluation matrix.
    acc = accuracy_score(labels, preds)
    prec = precision_score(labels, preds, average="macro", zero_division=0)
    rec = recall_score(labels, preds, average="macro", zero_division=0)
    f1 = f1_score(labels, preds, average="macro", zero_division=0)
    metrics = {
        "Precision": round(prec * 100, 2),
        "Recall": round(rec * 100, 2),
        "F1 Score": round(f1 * 100, 2),
        "Accuracy": round(acc * 100, 2),
    }
    if probs is not None:
        try:
            auc = roc_auc_score(labels, probs)
            metrics["AUC"] = round(auc * 100, 2)
        except ValueError:
            metrics["AUC"] = float("nan")
    return metrics


def find_best_threshold(labels, probs):
    # Youden's J statistic (tpr - fpr) on the VALIDATION set only -- never touch the
    # test set when choosing this threshold, or the reported test accuracy would be
    # optimistically biased (data leakage). This is a standard, legitimate technique
    # for imbalanced binary classification where AUC is high but the naive 0.5 cutoff
    # is not where sensitivity/specificity are actually balanced.
    fpr, tpr, thresholds = roc_curve(labels, probs)
    j_scores = tpr - fpr
    best_idx = j_scores.argmax()
    return float(thresholds[best_idx])


def preds_from_threshold(probs, threshold):
    return [1 if p >= threshold else 0 for p in probs]

print("Metric computation ready (Accuracy, Precision, Recall, F1, AUC, "
      "+ validation-based optimal threshold selection).")


## 9. Visualizing NISE Augmentations (Figure 2 equivalent)

Sanity-check the augmentation pipelines on a real chest X-ray before running the full pipeline.


In [ ]:

import matplotlib.pyplot as plt

def unnormalize(img_tensor):
    mean = torch.tensor(NORM_MEAN).view(3, 1, 1)
    std = torch.tensor(NORM_STD).view(3, 1, 1)
    return (img_tensor * std + mean).clamp(0, 1)

def show_augmentations(file_path, n_aug=4):
    img = Image.open(file_path)
    fig, axes = plt.subplots(1, n_aug + 1, figsize=(3 * (n_aug + 1), 3))
    axes[0].imshow(to_rgb(img))
    axes[0].set_title("Original")
    axes[0].axis("off")
    for i in range(n_aug):
        aug_img = simclr_transform(img)
        axes[i + 1].imshow(unnormalize(aug_img).permute(1, 2, 0).numpy())
        axes[i + 1].set_title(f"Augmentation {i+1}")
        axes[i + 1].axis("off")
    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG["OUTPUT_DIR"], "figure2_augmentations.png"), dpi=150)
    plt.show()

if len(train_files) > 0:
    show_augmentations(train_files[0])
else:
    print("No files found yet - run the dataset preparation cells first.")

# Auto-sync figures
for f in os.listdir(CONFIG['OUTPUT_DIR']):
    if f.endswith(('.png', '.pdf')):
        sync_file_to_gdrive(os.path.join(CONFIG['OUTPUT_DIR'], f), f)



## 10. Full Experiment Orchestration

Runs, for each backbone (`ResNet-50`, `EfficientNet-B0`) x each SSL method (`SimCLR`, `MoCo`, `BYOL`, `DINO`):

1. SSL pretraining (records per-epoch loss curve -> Figure 4/5/6/DINO-curve equivalents)
2. Two-stage fine-tuning, capturing:
   - **"Without fine-tuning"** metrics (end of Stage 1, frozen backbone) -> Table 3
   - **Fine-tuned** metrics at the configured checkpoints -> Table 4 / Table 5
   - Final **test-set** metrics with AUC -> Table 6
   - Train/val accuracy & loss curves -> Figure 7 equivalent

All raw results are stored in `RESULTS` for later table/figure generation.


### Checkpointing & resume (important for Kaggle)

This run is fully resumable at two levels:

1. **Combo-level**: once a `(backbone, method)` pair finishes completely, its results are
   saved to `outputs/results_checkpoint.pt`. Re-running this cell reloads that file and
   **skips any combo already completed**.
2. **Epoch-level**: within an unfinished combo, both SSL pretraining and the two-stage
   fine-tuning save model + optimizer state after **every single epoch** to
   `outputs/checkpoints/`. If a Kaggle session disconnects or times out mid-combo,
   re-running this cell resumes from the exact epoch (and exact stage, for fine-tuning)
   it left off at -- no epoch is redone.

You never need to change anything to get this behavior; just re-run the notebook (or the
orchestration cell) after a restart and it will pick up where it stopped.

In [ ]:
BACKBONES = ["resnet50", "efficientnet_b0"]
SSL_METHODS = ["SimCLR", "MoCo", "BYOL", "DINO"]  # DINO added per request

RESULTS_CKPT_PATH = os.path.join(CONFIG["OUTPUT_DIR"], "results_checkpoint.pt")

# ---- Load any already-completed combos so we don't redo finished work ----
if os.path.exists(RESULTS_CKPT_PATH):
    RESULTS = torch.load(RESULTS_CKPT_PATH, map_location="cpu", weights_only=False)
    print(RESULTS.keys())
    done_combos = [(b, m) for b in RESULTS for m in RESULTS[b]]
    print(f"Loaded RESULTS checkpoint - {len(done_combos)} combo(s) already completed: {done_combos}")
else:
    RESULTS = {}  # RESULTS[backbone][method] = {"ssl_loss": [...], "checkpoint_metrics": {...}, "history": {...}}

for backbone_name in BACKBONES:
    RESULTS.setdefault(backbone_name, {})
    for method_name in SSL_METHODS:
        if method_name in RESULTS[backbone_name]:
            print(f"Skipping {method_name}/{backbone_name} - already completed (found in RESULTS checkpoint).")
            continue

        print(f"\n{'='*70}\nRunning {method_name} with {backbone_name}\n{'='*70}")
        t0 = time.time()

        # ---- 1. SSL Pretraining (resumes from last saved epoch if a checkpoint exists) ----
        pretrain_fn = PRETRAIN_FUNCS[method_name]
        pretrained_backbone, ssl_loss_history = pretrain_fn(
            backbone_name, train_files, CONFIG["SSL_EPOCHS"], CONFIG["BATCH_SIZE_SSL"]
        )

        # ---- 2. Two-stage fine-tuning (resumes from last saved epoch/stage if a checkpoint exists) ----
        feature_dim = build_backbone(backbone_name)[1]
        eval_checkpoints = sorted(set([CONFIG["FT_STAGE1_EPOCHS"]] + CONFIG["FT_EVAL_EPOCHS"]))
        checkpoint_metrics, history = two_stage_finetune(
            pretrained_backbone, feature_dim, backbone_name, method_name,
            stage1_epochs=CONFIG["FT_STAGE1_EPOCHS"],
            stage2_epochs=CONFIG["FT_STAGE2_EPOCHS"],
            eval_epoch_checkpoints=eval_checkpoints,
            batch_size=CONFIG["BATCH_SIZE_CLF"],
            lr_head=CONFIG["LR_HEAD"],
            lr_finetune=CONFIG["LR_FINETUNE"],
        )

        RESULTS[backbone_name][method_name] = {
            "ssl_loss": ssl_loss_history,
            "checkpoint_metrics": checkpoint_metrics,
            "history": history,
        }

        # ---- Save combo-level checkpoint immediately so this combo never needs to rerun ----
        torch.save(RESULTS, RESULTS_CKPT_PATH)
        print(f"Finished {method_name}/{backbone_name} in {time.time()-t0:.1f}s - RESULTS checkpoint saved.")
        del pretrained_backbone, checkpoint_metrics, history
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print("\nAll experiments complete.")



## 11. Table Generation

Reproduces the paper's table structure (Tables 3–7), adapted to this experiment
(pneumonia, 2 classes, 4 SSL methods including the added DINO).


In [ ]:

import pandas as pd

STAGE1_EPOCH = CONFIG["FT_STAGE1_EPOCHS"]

# ---- Table 3 equivalent: comparison WITHOUT fine-tuning (ResNet-50) ----
rows = []
for method_name in SSL_METHODS:
    m = RESULTS["resnet50"][method_name]["checkpoint_metrics"].get(STAGE1_EPOCH, {})
    rows.append({"Approach": method_name, "Epochs": STAGE1_EPOCH, **m})
table3 = pd.DataFrame(rows)[["Approach", "Epochs", "Precision", "Recall", "F1 Score", "Accuracy"]]
print("Table 3 - Without fine-tuning (ResNet-50 backbone):")
display(table3)
table3.to_csv(os.path.join(CONFIG["OUTPUT_DIR"], "table3_without_finetuning_resnet50.csv"), index=False)

# Auto-sync saved tables
for f in os.listdir(CONFIG['OUTPUT_DIR']):
    if f.endswith('.csv'): sync_file_to_gdrive(os.path.join(CONFIG['OUTPUT_DIR'], f), f)


In [ ]:
# ---- Table 4 equivalent: fine-tuned comparison, ResNet-50, at each eval checkpoint ----
rows = []

for method_name in SSL_METHODS:
    metrics = RESULTS["resnet50"][method_name]["checkpoint_metrics"]

    # শুধুমাত্র integer epoch key গুলো নাও
    epoch_keys = sorted([k for k in metrics.keys() if isinstance(k, int)])

    for epoch in epoch_keys:
        m = metrics[epoch]
        rows.append({
            "Approach": method_name,
            "Epochs": epoch,
            "Precision": m["Precision"],
            "Recall": m["Recall"],
            "F1 Score": m["F1 Score"],
            "Accuracy": m["Accuracy"]
        })

table4 = pd.DataFrame(rows)

print("Table 4 - Fine-tuned comparison (ResNet-50 backbone):")
display(table4)

table4.to_csv(
    os.path.join(CONFIG["OUTPUT_DIR"], "table4_finetuned_resnet50.csv"),
    index=False
)
# Auto-sync saved tables
for f in os.listdir(CONFIG['OUTPUT_DIR']):
    if f.endswith('.csv'): sync_file_to_gdrive(os.path.join(CONFIG['OUTPUT_DIR'], f), f)


In [ ]:

# # ---- Table 4 equivalent: fine-tuned comparison, ResNet-50, at each eval checkpoint ----
# rows = []
# for method_name in SSL_METHODS:
#     for epoch, m in sorted(RESULTS["resnet50"][method_name]["checkpoint_metrics"].items()):
#         rows.append({"Approach": method_name, "Epochs": epoch, **m})
# table4 = pd.DataFrame(rows)[["Approach", "Epochs", "Precision", "Recall", "F1 Score", "Accuracy"]]
# print("Table 4 - Fine-tuned comparison (ResNet-50 backbone):")
# display(table4)
# table4.to_csv(os.path.join(CONFIG["OUTPUT_DIR"], "table4_finetuned_resnet50.csv"), index=False)

# Auto-sync saved tables
for f in os.listdir(CONFIG['OUTPUT_DIR']):
    if f.endswith('.csv'): sync_file_to_gdrive(os.path.join(CONFIG['OUTPUT_DIR'], f), f)


In [ ]:
# ---- Table 5 equivalent: fine-tuned comparison, EfficientNet-B0, at each eval checkpoint ----

rows = []

for method_name in SSL_METHODS:
    metrics = RESULTS["efficientnet_b0"][method_name]["checkpoint_metrics"]

    # শুধুমাত্র integer epoch key গুলো নাও
    epoch_keys = sorted([k for k in metrics.keys() if isinstance(k, int)])

    for epoch in epoch_keys:
        m = metrics[epoch]
        rows.append({
            "Approach": method_name,
            "Epochs": epoch,
            "Precision": m["Precision"],
            "Recall": m["Recall"],
            "F1 Score": m["F1 Score"],
            "Accuracy": m["Accuracy"]
        })

table5 = pd.DataFrame(rows)

print("Table 5 - Fine-tuned comparison (EfficientNet-B0 backbone):")
display(table5)

table5.to_csv(
    os.path.join(CONFIG["OUTPUT_DIR"], "table5_finetuned_efficientnet.csv"),
    index=False
)
# Auto-sync saved tables
for f in os.listdir(CONFIG['OUTPUT_DIR']):
    if f.endswith('.csv'): sync_file_to_gdrive(os.path.join(CONFIG['OUTPUT_DIR'], f), f)


In [ ]:

# # ---- Table 5 equivalent: fine-tuned comparison, EfficientNet-B0, at each eval checkpoint ----
# rows = []
# for method_name in SSL_METHODS:
#     for epoch, m in sorted(RESULTS["efficientnet_b0"][method_name]["checkpoint_metrics"].items()):
#         rows.append({"Approach": method_name, "Epochs": epoch, **m})
# table5 = pd.DataFrame(rows)[["Approach", "Epochs", "Precision", "Recall", "F1 Score", "Accuracy"]]
# print("Table 5 - Fine-tuned comparison (EfficientNet-B0 backbone):")
# display(table5)
# table5.to_csv(os.path.join(CONFIG["OUTPUT_DIR"], "table5_finetuned_efficientnet.csv"), index=False)

# Auto-sync saved tables
for f in os.listdir(CONFIG['OUTPUT_DIR']):
    if f.endswith('.csv'): sync_file_to_gdrive(os.path.join(CONFIG['OUTPUT_DIR'], f), f)


In [ ]:
# ---- Table 6 equivalent: final test-set summary (with AUC) for both backbones ----
# Also includes a "Best (val-selected + TTA + threshold)" row per combo -- the checkpoint
# picked by validation accuracy, evaluated with test-time augmentation and a decision
# threshold tuned on the validation set only (never on test data).
final_epoch = max(CONFIG["FT_EVAL_EPOCHS"])
rows = []
for backbone_name in BACKBONES:
    for method_name in SSL_METHODS:
        cm = RESULTS[backbone_name][method_name]["checkpoint_metrics"]
        m = cm.get(final_epoch, {})
        rows.append({"Approach": method_name, "Backbone": backbone_name, "Variant": f"epoch {final_epoch}", **m})
        if "best" in cm:
            best_m = cm["best"]
            rows.append({"Approach": method_name, "Backbone": backbone_name,
                         "Variant": f"best (ep {best_m.get('Selected epoch','?')}, TTA+thresh)", **best_m})

table6 = pd.DataFrame(rows)
cols = ["Approach", "Backbone", "Variant", "Precision", "Recall", "F1 Score", "Accuracy", "AUC"]
cols = [c for c in cols if c in table6.columns]
table6 = table6[cols]
print("Table 6 - Summary of SSL approaches on the held-out test set:")
display(table6)
table6.to_csv(os.path.join(CONFIG["OUTPUT_DIR"], "table6_test_summary.csv"), index=False)
# Auto-sync saved tables
for f in os.listdir(CONFIG['OUTPUT_DIR']):
    if f.endswith('.csv'): sync_file_to_gdrive(os.path.join(CONFIG['OUTPUT_DIR'], f), f)



## 11a. Supervised Baselines (Table 2) — CNN / ResNet50 / EfficientNet-B0 trained from scratch

Assumptions used here (flagging these since the source doc didn't fully specify them):
- **"From scratch"** is taken literally: ResNet-50 and EfficientNet-B0 are randomly initialized
  (`weights=None`, no ImageNet), then trained end-to-end — no SSL pretraining involved.
- **CNN** is a small custom 5-block conv net (defined below) since no architecture was specified.
- Each baseline is trained for `CONFIG["BASELINE_EPOCHS"]` epochs (default 30, matching the SSL
  fine-tuning budget of stage1+stage2), with the same class-weighted loss, label smoothing, and
  best-checkpoint + TTA + validation-tuned-threshold evaluation used for the SSL methods, so the
  numbers in Table 2 are directly comparable to Tables 3-7.


In [ ]:

# ---- Simple custom CNN backbone (no architecture was specified in the source doc) ----
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        def block(c_in, c_out):
            return nn.Sequential(
                nn.Conv2d(c_in, c_out, 3, padding=1),
                nn.BatchNorm2d(c_out),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
            )
        self.features = nn.Sequential(
            block(3, 32),
            block(32, 64),
            block(64, 128),
            block(128, 256),
            nn.Conv2d(256, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
        )
        self.feature_dim = 256

    def forward(self, x):
        return self.features(x).flatten(1)


def build_backbone_scratch(name):
    # Returns (backbone_module, feature_dim), randomly initialized (no ImageNet, no SSL).
    if name == "CNN":
        net = SimpleCNN()
        feature_dim = net.feature_dim
    elif name == "resnet50":
        net = models.resnet50(weights=None)
        feature_dim = net.fc.in_features
        net.fc = nn.Identity()
    elif name == "efficientnet_b0":
        net = models.efficientnet_b0(weights=None)
        feature_dim = net.classifier[1].in_features
        net.classifier = nn.Identity()
    else:
        raise ValueError(f"Unknown baseline backbone: {name}")

    net = net.to(DEVICE)
    if NUM_GPUS > 1:
        net = nn.DataParallel(net)
    return net, feature_dim

print("Baseline (from-scratch) backbone factory ready: CNN, resnet50, efficientnet_b0")


In [ ]:

# ---- Generic supervised training loop (single stage, everything trainable from epoch 1) ----
# Reused for: Table 2 baselines, Table 8 label-fraction sweep, and Table 9 ablation A1/A2.

def train_classifier_full(backbone, feature_dim, train_loader, val_loader, test_loader,
                           epochs, lr, tag, class_weights=None, label_smoothing=0.1,
                           ckpt_path=None, freeze_backbone=False):
    model = FullClassifier(backbone, feature_dim, CONFIG["NUM_CLASSES"]).to(DEVICE)
    if freeze_backbone:
        set_backbone_frozen(model.backbone, frozen=True, unfreeze_last_n_blocks=0)
        trainable_params = model.head.parameters()
    else:
        trainable_params = model.parameters()
    optimizer = torch.optim.Adam(trainable_params, lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, epochs))

    start_epoch = 0
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_val_acc, best_epoch, best_model_state = -1.0, -1, None

    if ckpt_path is not None and os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=DEVICE)
        model.load_state_dict(ckpt["model_state"])
        optimizer.load_state_dict(ckpt["optimizer_state"])
        scheduler.load_state_dict(ckpt["scheduler_state"])
        start_epoch = ckpt["epoch"] + 1
        history = ckpt["history"]
        best_val_acc = ckpt["best_val_acc"]
        best_epoch = ckpt["best_epoch"]
        best_model_state = ckpt["best_model_state"]
        print(f"[{tag}] resuming from epoch {start_epoch+1}/{epochs} (best val_acc {best_val_acc:.4f} @ {best_epoch})")

    for epoch in range(start_epoch, epochs):
        train_loss, train_labels_, train_preds, _ = run_epoch(
            model, train_loader, optimizer, class_weights=class_weights, label_smoothing=label_smoothing)
        val_loss, val_labels_, val_preds, _ = run_epoch(
            model, val_loader, optimizer=None, class_weights=class_weights, label_smoothing=label_smoothing)
        scheduler.step()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        gc.collect()
        train_acc = accuracy_score(train_labels_, train_preds)
        val_acc = accuracy_score(val_labels_, val_preds)
        history["train_loss"].append(train_loss); history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc);   history["val_acc"].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc, best_epoch = val_acc, epoch + 1
            best_model_state = copy.deepcopy(model.state_dict())

        print(f"[{tag}] epoch {epoch+1}/{epochs} - train_loss {train_loss:.4f} val_loss {val_loss:.4f} "
              f"train_acc {train_acc:.4f} val_acc {val_acc:.4f} (best {best_val_acc:.4f} @ {best_epoch})")

        if ckpt_path is not None:
            sync_file_to_gdrive(ckpt_path, os.path.relpath(ckpt_path, CONFIG["OUTPUT_DIR"]))
            torch.save({
                "epoch": epoch, "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(), "scheduler_state": scheduler.state_dict(),
                "history": history, "best_val_acc": best_val_acc, "best_epoch": best_epoch,
                "best_model_state": best_model_state,
            }, ckpt_path)

    # Final evaluation: best-val-accuracy checkpoint + TTA + validation-tuned threshold
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    val_labels_best, val_probs_best = run_epoch_tta(model, val_loader)
    best_threshold = find_best_threshold(val_labels_best, val_probs_best)
    test_labels_best, test_probs_best = run_epoch_tta(model, test_loader)
    test_preds_best = preds_from_threshold(test_probs_best, best_threshold)
    metrics = compute_metrics(test_labels_best, test_preds_best, test_probs_best)
    metrics["Selected epoch"] = best_epoch
    metrics["Threshold"] = round(best_threshold, 4)
    print(f"[{tag}] FINAL (best ep {best_epoch}, TTA+threshold): {metrics}")
    return metrics, history

print("Generic supervised training loop ready (train_classifier_full).")


In [ ]:

# ---- Run the 3 from-scratch baselines (Table 2) ----
BASELINE_NAMES = ["CNN", "resnet50", "efficientnet_b0"]
BASELINE_RESULTS_PATH = os.path.join(CONFIG["OUTPUT_DIR"], "baseline_results.pt")

if os.path.exists(BASELINE_RESULTS_PATH):
    BASELINE_RESULTS = torch.load(BASELINE_RESULTS_PATH, map_location="cpu", weights_only=False)
    print(f"Loaded baseline results checkpoint - {list(BASELINE_RESULTS.keys())} already completed")
else:
    BASELINE_RESULTS = {}

train_loader, val_loader, test_loader = make_clf_loaders(CONFIG["BATCH_SIZE_CLF"])
class_counts = np.bincount(train_labels, minlength=CONFIG["NUM_CLASSES"]).astype(float)
class_weights = torch.tensor(
    len(train_labels) / (CONFIG["NUM_CLASSES"] * np.maximum(class_counts, 1)),
    dtype=torch.float32, device=DEVICE)

for name in BASELINE_NAMES:
    if name in BASELINE_RESULTS:
        print(f"Skipping baseline {name} - already completed.")
        continue
    backbone, feature_dim = build_backbone_scratch(name)
    ckpt_path = os.path.join(CHECKPOINT_DIR, f"baseline_{name}.pt")
    metrics, history = train_classifier_full(
        backbone, feature_dim, train_loader, val_loader, test_loader,
        epochs=CONFIG["BASELINE_EPOCHS"], lr=CONFIG["LR_FINETUNE"], tag=f"baseline:{name}",
        class_weights=class_weights, ckpt_path=ckpt_path)
    BASELINE_RESULTS[name] = {"metrics": metrics, "history": history}
    torch.save(BASELINE_RESULTS, BASELINE_RESULTS_PATH)

print("\nAll baseline experiments complete.")


In [ ]:

# ---- Table 2: Performance Comparison of Supervised Learning Models (Baseline Experiment) ----
DISPLAY_NAME = {"CNN": "CNN", "resnet50": "ResNet50", "efficientnet_b0": "EfficientNet-B0"}
rows = []
for i, name in enumerate(BASELINE_NAMES, start=1):
    m = BASELINE_RESULTS[name]["metrics"]
    rows.append({
        "Experiment ID": f"S{i}", "Feature Learning Strategy": "From scratch",
        "Classification Model": DISPLAY_NAME[name],
        "Accuracy (%)": m["Accuracy"], "Precision (%)": m["Precision"],
        "Recall(%)": m["Recall"], "F1-score (%)": m["F1 Score"], "AUC": m.get("AUC", float("nan")),
    })
table2 = pd.DataFrame(rows)
print("Table 2 - Performance Comparison of Supervised Learning Models (Baseline Experiment):")
display(table2)
table2.to_csv(os.path.join(CONFIG["OUTPUT_DIR"], "table2_supervised_baselines.csv"), index=False)

# Auto-sync saved tables
for f in os.listdir(CONFIG['OUTPUT_DIR']):
    if f.endswith('.csv'): sync_file_to_gdrive(os.path.join(CONFIG['OUTPUT_DIR'], f), f)



## 11b. Table 7 — Summary of All Approaches on the Held-Out Test Set

Table 6 plus the three from-scratch supervised baselines appended (Backbone shown as "-" for
these, matching the source doc, since they aren't paired with a separate SSL backbone role).


In [ ]:

rows = table6.copy()
rows = rows.to_dict("records")
for name in BASELINE_NAMES:
    m = BASELINE_RESULTS[name]["metrics"]
    rows.append({
        "Approach": DISPLAY_NAME[name], "Backbone": "-",
        "Precision": m["Precision"], "Recall": m["Recall"],
        "F1 Score": m["F1 Score"], "Accuracy": m["Accuracy"], "AUC": m.get("AUC", float("nan")),
    })
table7 = pd.DataFrame(rows)
print("Table 7 - Summary of all approaches on the held-out test set:")
display(table7)
table7.to_csv(os.path.join(CONFIG["OUTPUT_DIR"], "table7_all_approaches_summary.csv"), index=False)

# Auto-sync saved tables
for f in os.listdir(CONFIG['OUTPUT_DIR']):
    if f.endswith('.csv'): sync_file_to_gdrive(os.path.join(CONFIG['OUTPUT_DIR'], f), f)



## 11c. Effect of Different Amounts of Labeled Data (Table 8)

Setup used here:
- **CNN / ResNet50 / EfficientNet-B0** (supervised, from scratch): retrained from random init using
  only a stratified X% subsample of the *training* labels (val/test sets are untouched).
- **SimCLR / MoCo / BYOL / DINO**: the backbone stays the one already SSL-pretrained on 100% of the
  (unlabeled) images in Section 10 — only the fine-tuning stage is restricted to X% of labels. This
  is the standard semi-supervised SSL setup and is what makes SSL methods interesting at low label
  fractions.
- All models are evaluated on the full test set; only the amount of *training* supervision varies.
- This cell is checkpointed per (fraction, model) pair, since the full grid (5 fractions x 7 models)
  is a lot of training — safe to interrupt and resume.


In [ ]:

def subsample_labeled(files, labels, fraction, seed=CONFIG["SEED"]):
    # Stratified subsample of the training set (keeps class balance roughly constant).
    if fraction >= 1.0:
        return list(files), list(labels)
    files = np.array(files); labels = np.array(labels)
    keep_idx = []
    rng = np.random.RandomState(seed)
    for c in np.unique(labels):
        c_idx = np.where(labels == c)[0]
        n_keep = max(1, int(round(len(c_idx) * fraction)))
        keep_idx.extend(rng.choice(c_idx, size=n_keep, replace=False))
    keep_idx = np.array(keep_idx)
    return files[keep_idx].tolist(), labels[keep_idx].tolist()


def make_custom_clf_loaders(files, labels, batch_size, train_transform, eval_transform):
    train_ds = LabeledImageDataset(files, labels, train_transform)
    val_ds   = LabeledImageDataset(val_files, val_labels, eval_transform)
    test_ds  = LabeledImageDataset(test_files, test_labels, eval_transform)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                               num_workers=CONFIG["NUM_WORKERS"], pin_memory=True, drop_last=False)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                               num_workers=CONFIG["NUM_WORKERS"], pin_memory=True)
    test_loader  = DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                               num_workers=CONFIG["NUM_WORKERS"], pin_memory=True)
    return train_loader, val_loader, test_loader


def class_weights_for(labels_subset):
    counts = np.bincount(labels_subset, minlength=CONFIG["NUM_CLASSES"]).astype(float)
    return torch.tensor(len(labels_subset) / (CONFIG["NUM_CLASSES"] * np.maximum(counts, 1)),
                         dtype=torch.float32, device=DEVICE)

print("Label-subsampling utilities ready.")


In [ ]:

LABEL_FRAC_RESULTS_PATH = os.path.join(CONFIG["OUTPUT_DIR"], "label_fraction_results.pt")
if os.path.exists(LABEL_FRAC_RESULTS_PATH):
    LABEL_FRAC_RESULTS = torch.load(LABEL_FRAC_RESULTS_PATH, map_location="cpu",  weights_only=False)
    print(f"Loaded label-fraction checkpoint - {len(LABEL_FRAC_RESULTS)} (fraction, model) combos done")
else:
    LABEL_FRAC_RESULTS = {}  # LABEL_FRAC_RESULTS[(frac, model_name)] = metrics dict

SUPERVISED_MODELS = ["CNN", "resnet50", "efficientnet_b0"]
SSL_MODEL_TO_BACKBONE = {  # which pretrained (backbone_name, RESULTS key) each SSL method uses here
    "SimCLR": "resnet50", "MoCo": "resnet50", "BYOL": "resnet50", "DINO": "resnet50",
}

for frac in CONFIG["LABELED_FRACTIONS"]:
    frac_files, frac_labels = subsample_labeled(train_files, train_labels, frac)
    cw = class_weights_for(frac_labels)

    # ---- Supervised baselines, retrained from scratch on the subsample ----
    for name in SUPERVISED_MODELS:
        key = (frac, name)
        if key in LABEL_FRAC_RESULTS:
            continue
        tr_loader, va_loader, te_loader = make_custom_clf_loaders(
            frac_files, frac_labels, CONFIG["BATCH_SIZE_CLF"], clf_train_transform, clf_eval_transform)
        backbone, feature_dim = build_backbone_scratch(name)
        tag = f"labelfrac:{frac}:{name}"
        ckpt_path = os.path.join(CHECKPOINT_DIR, f"labelfrac_{frac}_{name}.pt")
        metrics, _ = train_classifier_full(
            backbone, feature_dim, tr_loader, va_loader, te_loader,
            epochs=CONFIG["BASELINE_EPOCHS"], lr=CONFIG["LR_FINETUNE"], tag=tag,
            class_weights=cw, ckpt_path=ckpt_path)
        LABEL_FRAC_RESULTS[key] = metrics
        torch.save(LABEL_FRAC_RESULTS, LABEL_FRAC_RESULTS_PATH)

    # ---- SSL methods: reuse the already-pretrained backbone, fine-tune on the subsample ----
    for method_name, backbone_name in SSL_MODEL_TO_BACKBONE.items():
        key = (frac, method_name)
        if key in LABEL_FRAC_RESULTS:
            continue
        # Get a fresh copy of the SSL-pretrained backbone. pretrain_* functions load their
        # per-epoch checkpoint and, since SSL_EPOCHS is already complete (Section 10 ran first),
        # this returns immediately without retraining - just deserializes the converged weights.
        feature_dim = build_backbone(backbone_name)[1]
        pretrain_fn = PRETRAIN_FUNCS[method_name]
        fresh_backbone, _ = pretrain_fn(backbone_name, train_files, CONFIG["SSL_EPOCHS"], CONFIG["BATCH_SIZE_SSL"])

        tr_loader, va_loader, te_loader = make_custom_clf_loaders(
            frac_files, frac_labels, CONFIG["BATCH_SIZE_CLF"], clf_train_transform, clf_eval_transform)
        tag = f"labelfrac:{frac}:{method_name}"
        ckpt_path = os.path.join(CHECKPOINT_DIR, f"labelfrac_{frac}_{method_name}.pt")
        metrics, _ = train_classifier_full(
            fresh_backbone, feature_dim, tr_loader, va_loader, te_loader,
            epochs=CONFIG["FT_STAGE1_EPOCHS"] + CONFIG["FT_STAGE2_EPOCHS"], lr=CONFIG["LR_FINETUNE"],
            tag=tag, class_weights=cw, ckpt_path=ckpt_path)
        LABEL_FRAC_RESULTS[key] = metrics
        torch.save(LABEL_FRAC_RESULTS, LABEL_FRAC_RESULTS_PATH)

print("\nLabel-fraction sweep complete.")


In [ ]:

# ---- Table 8: Effect of Different Amounts of Labeled Data (Accuracy %, pivoted) ----
COLS_ORDER = ["CNN", "resnet50", "efficientnet_b0", "SimCLR", "MoCo", "BYOL", "DINO"]
COL_DISPLAY = {"CNN": "CNN", "resnet50": "ResNet50", "efficientnet_b0": "EfficientNet",
               "SimCLR": "SimCLR", "MoCo": "MoCo", "BYOL": "BYOL", "DINO": "DINO"}

rows = []
for frac in CONFIG["LABELED_FRACTIONS"]:
    row = {"Labeled Data (%)": f"{int(frac*100)}% labeled"}
    for col in COLS_ORDER:
        m = LABEL_FRAC_RESULTS.get((frac, col), {})
        row[COL_DISPLAY[col]] = m.get("Accuracy", float("nan"))
    rows.append(row)
table8 = pd.DataFrame(rows)
print("Table 8 - Effect of Different Amounts of Labeled Data (Accuracy %):")
display(table8)
table8.to_csv(os.path.join(CONFIG["OUTPUT_DIR"], "table8_labeled_data_effect.csv"), index=False)

# Auto-sync saved tables
for f in os.listdir(CONFIG['OUTPUT_DIR']):
    if f.endswith('.csv'): sync_file_to_gdrive(os.path.join(CONFIG['OUTPUT_DIR'], f), f)



## 11d. Ablation Study of SSL Components (Table 9)

The source table lists "DINO" pretraining for all four rows but A1/A2 both say "No augmentation"
and A3/A4 both say "Strong augmentation" verbatim — that duplication looks like a data-entry
artifact rather than 4 distinct configs. To make the ablation actually informative (and match the
row count/labels in the doc), it's implemented as a proper 4-way ablation on **ResNet-50**:

| Exp | Pretraining | Augmentation (fine-tune stage) | Fine-tuning |
|---|---|---|---|
| A1 | None (random init) | None | Yes |
| A2 | DINO | None | Yes |
| A3 | DINO | Light (`clf_eval_transform`-based light jitter) | Yes |
| A4 | DINO | Strong (full NISE `clf_train_transform` pipeline) | Yes |

A1→A2 isolates the effect of SSL pretraining; A2→A3→A4 isolates the effect of augmentation
strength during fine-tuning. Adjust the table above if a different split was intended.


In [ ]:

# A "light" augmentation variant for A3 (between "none" and the full NISE pipeline used elsewhere)
clf_light_aug_transform = T.Compose([
    T.Lambda(to_rgb),
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.ToTensor(),
    T.Normalize(NORM_MEAN, NORM_STD),
])

ABLATION_CONFIGS = {
    "A1": {"pretraining": None,   "augmentation": "none",   "train_transform": clf_eval_transform},
    "A2": {"pretraining": "DINO", "augmentation": "none",   "train_transform": clf_eval_transform},
    "A3": {"pretraining": "DINO", "augmentation": "light",  "train_transform": clf_light_aug_transform},
    "A4": {"pretraining": "DINO", "augmentation": "strong", "train_transform": clf_train_transform},
}
ABLATION_BACKBONE = "resnet50"

ABLATION_RESULTS_PATH = os.path.join(CONFIG["OUTPUT_DIR"], "ablation_results.pt")
if os.path.exists(ABLATION_RESULTS_PATH):
    ABLATION_RESULTS = torch.load(ABLATION_RESULTS_PATH, map_location="cpu", weights_only=False)
    print(f"Loaded ablation checkpoint - {list(ABLATION_RESULTS.keys())} already completed")
else:
    ABLATION_RESULTS = {}

cw_full = class_weights_for(train_labels)
tr_eval_loader, va_loader, te_loader = make_custom_clf_loaders(
    train_files, train_labels, CONFIG["BATCH_SIZE_CLF"], clf_eval_transform, clf_eval_transform)

for exp_id, cfg in ABLATION_CONFIGS.items():
    if exp_id in ABLATION_RESULTS:
        print(f"Skipping {exp_id} - already completed.")
        continue

    if cfg["pretraining"] is None:
        backbone, feature_dim = build_backbone_scratch(ABLATION_BACKBONE)
    else:
        # feature_dim is fixed by architecture (2048 for resnet50, 1280 for efficientnet_b0),
        # independent of weights, so it's safe to read it off a fresh build_backbone() call.
        feature_dim = build_backbone(ABLATION_BACKBONE)[1]
        pretrain_fn = PRETRAIN_FUNCS[cfg["pretraining"]]
        backbone, _ = pretrain_fn(
            ABLATION_BACKBONE, train_files, CONFIG["SSL_EPOCHS"], CONFIG["BATCH_SIZE_SSL"])

    tr_loader, _, _ = make_custom_clf_loaders(
        train_files, train_labels, CONFIG["BATCH_SIZE_CLF"], cfg["train_transform"], clf_eval_transform)

    ckpt_path = os.path.join(CHECKPOINT_DIR, f"ablation_{exp_id}.pt")
    metrics, _ = train_classifier_full(
        backbone, feature_dim, tr_loader, va_loader, te_loader,
        epochs=CONFIG["BASELINE_EPOCHS"], lr=CONFIG["LR_FINETUNE"], tag=f"ablation:{exp_id}",
        class_weights=cw_full, ckpt_path=ckpt_path)
    ABLATION_RESULTS[exp_id] = metrics
    torch.save(ABLATION_RESULTS, ABLATION_RESULTS_PATH)

print("\nAblation study complete.")


In [ ]:

# ---- Table 9: Ablation Study of SSL Components ----
rows = []
for exp_id, cfg in ABLATION_CONFIGS.items():
    m = ABLATION_RESULTS[exp_id]
    rows.append({
        "Experiment": exp_id,
        "Pretraining": cfg["pretraining"] or "None",
        "Augmentation": {"none": "No", "light": "Light augmentation", "strong": "Strong augmentation"}[cfg["augmentation"]],
        "Fine-tuning": "Yes",
        "Accuracy (%)": m["Accuracy"],
        "AUC": m.get("AUC", float("nan")),
    })
table9 = pd.DataFrame(rows)
print("Table 9 - Ablation Study of SSL Components:")
display(table9)
table9.to_csv(os.path.join(CONFIG["OUTPUT_DIR"], "table9_ablation_study.csv"), index=False)

# Auto-sync saved tables
for f in os.listdir(CONFIG['OUTPUT_DIR']):
    if f.endswith('.csv'): sync_file_to_gdrive(os.path.join(CONFIG['OUTPUT_DIR'], f), f)



## 12. Figures — SSL Loss Curves (Figure 4/5/6 equivalents)

One loss curve per SSL method, comparing ResNet-50 vs. EfficientNet-B0, exactly like the paper's
NT-Xent / InfoNCE / MSE loss curve figures — plus a new curve for DINO.


In [ ]:

def plot_ssl_loss_curves():
    fig, axes = plt.subplots(1, len(SSL_METHODS), figsize=(6 * len(SSL_METHODS), 4))
    if len(SSL_METHODS) == 1:
        axes = [axes]
    for ax, method_name in zip(axes, SSL_METHODS):
        for backbone_name, color in zip(BACKBONES, ["tab:blue", "tab:red"]):
            loss_hist = RESULTS[backbone_name][method_name]["ssl_loss"]
            ax.plot(range(1, len(loss_hist) + 1), loss_hist, marker="o", markersize=3,
                    label=backbone_name, color=color)
        ax.set_title(f"{method_name} pretraining loss")
        ax.set_xlabel("No. of Epochs")
        ax.set_ylabel("Loss")
        ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG["OUTPUT_DIR"], "figure_ssl_loss_curves.png"), dpi=150)
    plt.show()

plot_ssl_loss_curves()

# Auto-sync figures
for f in os.listdir(CONFIG['OUTPUT_DIR']):
    if f.endswith(('.png', '.pdf')):
        sync_file_to_gdrive(os.path.join(CONFIG['OUTPUT_DIR'], f), f)



## 13. Figures — Training vs. Validation Accuracy & Loss (Figure 7 equivalent)

Grid of curves: rows = backbone, columns = SSL method; each cell shows train/val accuracy
(top) and train/val loss (bottom) across the full two-stage fine-tuning run.


In [ ]:

def plot_finetune_curves():
    n_methods = len(SSL_METHODS)
    n_backbones = len(BACKBONES)
    fig, axes = plt.subplots(n_backbones * 2, n_methods, figsize=(5 * n_methods, 4 * n_backbones * 2))

    for bi, backbone_name in enumerate(BACKBONES):
        for mi, method_name in enumerate(SSL_METHODS):
            hist = RESULTS[backbone_name][method_name]["history"]
            epochs = range(1, len(hist["train_acc"]) + 1)

            ax_acc = axes[bi * 2, mi]
            ax_acc.plot(epochs, hist["train_acc"], label="Train Acc", color="tab:blue")
            ax_acc.plot(epochs, hist["val_acc"], label="Val Acc", color="tab:green")
            ax_acc.set_title(f"{backbone_name} | {method_name}")
            ax_acc.set_ylabel("Accuracy")
            ax_acc.legend(fontsize=8)

            ax_loss = axes[bi * 2 + 1, mi]
            ax_loss.plot(epochs, hist["train_loss"], label="Train Loss", color="tab:blue")
            ax_loss.plot(epochs, hist["val_loss"], label="Val Loss", color="tab:orange")
            ax_loss.set_xlabel("No. of Epochs")
            ax_loss.set_ylabel("Loss")
            ax_loss.legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG["OUTPUT_DIR"], "figure7_finetune_curves.png"), dpi=150)
    plt.show()

plot_finetune_curves()

# Auto-sync figures
for f in os.listdir(CONFIG['OUTPUT_DIR']):
    if f.endswith(('.png', '.pdf')):
        sync_file_to_gdrive(os.path.join(CONFIG['OUTPUT_DIR'], f), f)



## 14. Summary & Notes

- Tables 2-9 are all generated in this notebook and saved as CSV files under `outputs/`.
- All figures are saved as PNG files under `outputs/`.
- Set `CONFIG["QUICK_TEST"] = False` at the top and re-run the whole notebook for
  paper-scale results (30 SSL pretraining epochs, 15+15 fine-tuning epochs, checkpoints at 10/20/30).
- A GPU is strongly recommended for the full run (the paper reports ~45 min per configuration
  on a Kaggle P100 for ResNet-50, ~30 min for EfficientNet-B0). On CPU-only environments,
  keep `QUICK_TEST = True` or reduce `IMG_SIZE` / epochs further.
- **Key difference from the original paper**: 4-class brain tumor classification -> 2-class
  pneumonia classification, and a 4th SSL method (**DINO**) has been added alongside SimCLR, MoCo, and BYOL.


In [ ]:

print("Notebook run complete.")
print("Backbones evaluated:", BACKBONES)
print("SSL methods evaluated:", SSL_METHODS)
print("Supervised baselines evaluated:", BASELINE_NAMES)
print("Tables generated: 2, 3, 4, 5, 6, 7, 8, 9 (CSVs under outputs/)")
print("Outputs saved to:", os.path.abspath(CONFIG["OUTPUT_DIR"]))
